# Feature Importance Heatmaps

Two heatmaps:
1. **Top 30 features** — Gain vs Split importance (normalized), single model on all training data
2. **Monthly drift** — how the top-15 feature importances shift month-to-month

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier

sns.set_theme(style='whitegrid', font_scale=0.9)

## 1. Load & preprocess

In [ ]:
train = pd.read_csv('../public_data/train.csv')

# Normalise categoricals (same as EDA notebook)
cat_cols = train.select_dtypes(include='object').columns.tolist()
norm_cols = [c for c in cat_cols if c not in ['CustomerID', 'Month', 'ChurnStatus']]
for col in norm_cols:
    train[col] = train[col].str.strip().str.lower().str.replace('-', ' ', regex=False)

print(train.shape)
train.head(3)

In [ ]:
DROP = ['CustomerID', 'Month', 'ChurnStatus']

y = (train['ChurnStatus'].str.strip().str.lower() == 'yes').astype(int)
X = train.drop(columns=DROP).copy()

# Add month_idx so engineered sin/cos features appear (matches pipeline)
month_order = sorted(train['Month'].unique(),
                     key=lambda m: pd.to_datetime(m, format='%y-%b'))
month_idx_map = {m: i for i, m in enumerate(month_order)}
X['month_idx'] = train['Month'].map(month_idx_map)

# Encode all remaining object columns as category codes
for col in X.select_dtypes(include='object').columns:
    X[col] = pd.Categorical(X[col]).codes

# Fill remaining NaNs with column medians
X = X.fillna(X.median(numeric_only=True))

print('Features:', X.shape[1])
print('Positive rate:', y.mean().round(4))

## 2. Train LightGBM (all months)

In [ ]:
lgbm_params = dict(
    objective='binary',
    is_unbalance=True,
    n_estimators=300,
    random_state=42,
    verbosity=-1,
)

model = LGBMClassifier(**lgbm_params)
model.fit(X, y)
print('Done — feature count:', len(model.feature_name_))

## 3. Heatmap 1 — Top 30 features: Gain vs Split (normalized)

In [ ]:
TOP_N = 30

gain  = model.booster_.feature_importance(importance_type='gain')
split = model.booster_.feature_importance(importance_type='split')
names = model.feature_name_

imp_df = pd.DataFrame({
    'Feature': names,
    'Gain':    gain,
    'Split':   split,
}).sort_values('Gain', ascending=False).head(TOP_N)

# Normalise each metric to [0, 1] independently so they're comparable on same colour scale
imp_df['Gain_norm']  = imp_df['Gain']  / imp_df['Gain'].max()
imp_df['Split_norm'] = imp_df['Split'] / imp_df['Split'].max()

heatmap_data = imp_df.set_index('Feature')[['Gain_norm', 'Split_norm']]
heatmap_data.columns = ['Gain (norm)', 'Split (norm)']

# Annotation: show raw values
annot_gain  = imp_df.set_index('Feature')['Gain'].map(lambda v: f'{v:,.0f}')
annot_split = imp_df.set_index('Feature')['Split'].map(lambda v: f'{v:,.0f}')
annot = pd.concat([annot_gain, annot_split], axis=1)
annot.columns = ['Gain (norm)', 'Split (norm)']

fig, ax = plt.subplots(figsize=(7, 13))
sns.heatmap(
    heatmap_data,
    annot=annot,
    fmt='',
    cmap='YlOrRd',
    linewidths=0.4,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Normalised importance (0–1)', 'shrink': 0.5},
    vmin=0, vmax=1,
)
ax.set_title(f'Top {TOP_N} Feature Importance — Gain vs Split', fontsize=13, pad=12)
ax.set_xlabel('')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.savefig('feature_importance_gain_split.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → feature_importance_gain_split.png')

## 4. Heatmap 2 — Monthly importance drift (top 15 features)

Train one LightGBM per month using leave-one-month-out style:
each model is trained on **all months except the target month**, then evaluated on the target month.
Feature importance is extracted from each per-month model to show how it shifts over time.

In [ ]:
TOP_MONTHLY = 15

# Use the top-15 features by overall Gain as the anchor set
top_features = imp_df['Feature'].tolist()[:TOP_MONTHLY]

month_order_sorted = sorted(
    train['Month'].unique(),
    key=lambda m: pd.to_datetime(m, format='%y-%b')
)

monthly_importance = {}  # month_label -> {feature: gain}

for month in month_order_sorted:
    mask   = train['Month'] == month
    X_m    = X[mask]
    y_m    = y[mask]

    if y_m.nunique() < 2 or len(y_m) < 50:
        # Skip months with too few samples or only one class
        continue

    m_model = LGBMClassifier(**lgbm_params)
    m_model.fit(X_m, y_m)

    g = m_model.booster_.feature_importance(importance_type='gain')
    monthly_importance[month] = dict(zip(m_model.feature_name_, g))

print(f'Trained {len(monthly_importance)} monthly models')

In [ ]:
# Build matrix: features (rows) × months (cols)
monthly_df = pd.DataFrame(monthly_importance).reindex(index=top_features).fillna(0)

# Normalise each month column independently so months with different scales are comparable
monthly_norm = monthly_df.div(monthly_df.max(axis=0).replace(0, 1), axis=1)

# Shorten month labels for readability (e.g. '25-Jan' → 'Jan')
short_labels = {
    m: pd.to_datetime(m, format='%y-%b').strftime('%b\'%y')
    for m in monthly_norm.columns
}
monthly_norm.columns = [short_labels.get(c, c) for c in monthly_norm.columns]

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    monthly_norm,
    cmap='Blues',
    linewidths=0.3,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Normalised Gain (per month)', 'shrink': 0.6},
    vmin=0, vmax=1,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 8},
)
ax.set_title(f'Monthly Feature Importance Drift — Top {TOP_MONTHLY} Features', fontsize=13, pad=12)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.savefig('feature_importance_monthly.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → feature_importance_monthly.png')

## 5. Summary table
Raw gain + split values for the top 30 features.

In [ ]:
summary = imp_df[['Feature', 'Gain', 'Split']].copy()
summary['Gain %'] = (summary['Gain'] / summary['Gain'].sum() * 100).round(2)
summary['Split %'] = (summary['Split'] / summary['Split'].sum() * 100).round(2)
summary = summary.reset_index(drop=True)
summary.index += 1
summary